In [2]:
from pathlib import Path
import json
import re
from collections import Counter
from bs4 import BeautifulSoup
import pandas as pd

In [3]:
BASE_DIR = Path("../matsne_data")
SEARCH_DIR = BASE_DIR / "search_pages"
RAW_DIR = BASE_DIR / "documents_raw"
OUTPUT_FILE = BASE_DIR / "documents.jsonl"
FAILED_FILE = BASE_DIR / "failed_documents.json"

# Load JSONL
docs = []

with OUTPUT_FILE.open("r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        
        try:
            docs.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"JSON error on line {line_number}: {e}")

print(f"Documents loaded: {len(docs):,}")

Documents loaded: 300


# Basic Checks

In [4]:
ids = [d.get("document_id") for d in docs]
titles = [d.get("title") for d in docs]
urls = [d.get("url") for d in docs]

print("=== BASIC INTEGRITY ===")
print(f"Documents:       {len(docs):,}")
print(f"Unique IDs:      {len(set(ids)):,}")
print(f"Duplicate IDs:   {len(ids) - len(set(ids)):,}")
print(f"Unique URLs:     {len(set(urls)):,}")
print(f"Duplicate URLs:  {len(urls) - len(set(urls)):,}")
print(f"Missing IDs:     {sum(x is None or x == '' for x in ids):,}")
print(f"Missing titles:  {sum(x is None or x == '' for x in titles):,}")
print(f"Missing URLs:    {sum(x is None or x == '' for x in urls):,}")

=== BASIC INTEGRITY ===
Documents:       300
Unique IDs:      300
Duplicate IDs:   0
Unique URLs:     300
Duplicate URLs:  0
Missing IDs:     0
Missing titles:  0
Missing URLs:    0


# Check Failures

In [5]:
if FAILED_FILE.exists():
    failed = json.loads(FAILED_FILE.read_text(encoding="utf-8"))
else:
    failed = []

print("=== DOWNLOAD FAILURES ===")
print(f"Failed documents: {len(failed)}")

if failed:
    display(pd.DataFrame(failed))

=== DOWNLOAD FAILURES ===
Failed documents: 0


# Check raw HTML - JSONL

In [6]:
json_ids = set(ids)

raw_ids = {
    path.stem
    for path in RAW_DIR.glob("*.html")
}

missing_raw = json_ids - raw_ids
extra_raw = raw_ids - json_ids

print("=== RAW HTML INTEGRITY ===")
print(f"JSON documents:       {len(json_ids):,}")
print(f"Raw HTML files:       {len(raw_ids):,}")
print(f"JSON IDs without raw: {len(missing_raw):,}")
print(f"Raw files without JSON: {len(extra_raw):,}")

if missing_raw:
    print("\nMissing raw HTML:")
    print(sorted(missing_raw)[:20])

if extra_raw:
    print("\nExtra raw HTML:")
    print(sorted(extra_raw)[:20])

=== RAW HTML INTEGRITY ===
JSON documents:       300
Raw HTML files:       300
JSON IDs without raw: 0
Raw files without JSON: 0


In [7]:
def extract_search_ids(path):
    html = path.read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "html.parser")
    
    result_items = soup.select("div.document-search-result-item")
    
    found = []
    
    for item in result_items:
        link = item.select_one(".panel-heading a")
        if not link:
            continue
        
        href = link.get("href", "")
        match = re.search(r"/document/view/(\d+)", href)
        
        if match:
            found.append(match.group(1))
    
    return found


search_ids_by_page = {}

for path in sorted(SEARCH_DIR.glob("page_*.html")):
    page_ids = extract_search_ids(path)
    page_number = int(re.search(r"page_(\d+)", path.name).group(1))
    search_ids_by_page[page_number] = page_ids
    
    print(
        f"Page {page_number}: "
        f"{len(page_ids)} results, "
        f"{len(set(page_ids))} unique IDs"
    )

search_ids = [
    document_id
    for page_ids in search_ids_by_page.values()
    for document_id in page_ids
]

search_id_set = set(search_ids)
json_id_set = set(ids)

print("\n=== SEARCH ↔ CORPUS ===")
print(f"Search result records: {len(search_ids):,}")
print(f"Unique search IDs:     {len(search_id_set):,}")
print(f"Corpus IDs:            {len(json_id_set):,}")

print(f"Search IDs missing from corpus: {len(search_id_set - json_id_set):,}")
print(f"Corpus IDs not in search:       {len(json_id_set - search_id_set):,}")

Page 1: 100 results, 100 unique IDs
Page 2: 100 results, 100 unique IDs
Page 3: 100 results, 100 unique IDs
Page 4: 100 results, 100 unique IDs
Page 5: 100 results, 100 unique IDs

=== SEARCH ↔ CORPUS ===
Search result records: 500
Unique search IDs:     500
Corpus IDs:            300
Search IDs missing from corpus: 200
Corpus IDs not in search:       0


# Metadata

In [8]:
metadata_fields = [
    "document_id",
    "title",
    "url",
    "language",
    "document_type",
    "issuing_authority",
    "document_number",
    "signing_date",
]

print("=== METADATA COMPLETENESS ===")

metadata_report = []

for field in metadata_fields:
    missing = sum(
        d.get(field) is None or str(d.get(field)).strip() == ""
        for d in docs
    )
    
    metadata_report.append({
        "field": field,
        "missing": missing,
        "complete": len(docs) - missing,
        "completeness_%": round(
            100 * (len(docs) - missing) / len(docs), 2
        )
    })

display(pd.DataFrame(metadata_report))

=== METADATA COMPLETENESS ===


,field,missing,complete,completeness_%
0,document_id,0,300,100.0
1,title,0,300,100.0
2,url,0,300,100.0
3,language,0,300,100.0
4,document_type,0,300,100.0
5,issuing_authority,0,300,100.0
6,document_number,0,300,100.0
7,signing_date,0,300,100.0


# Length

In [9]:
text_lengths = []

for d in docs:
    text = d.get("text") or ""
    
    text_lengths.append({
        "document_id": d["document_id"],
        "title": d["title"],
        "characters": len(text),
        "words": len(text.split()),
        "parts": d.get("part_count", 0),
    })

text_df = pd.DataFrame(text_lengths)

print("=== TEXT INTEGRITY ===")
print(f"Empty documents:        {(text_df['characters'] == 0).sum()}")
print(f"<100 characters:        {(text_df['characters'] < 100).sum()}")
print(f"<500 characters:        {(text_df['characters'] < 500).sum()}")
print(f"<1,000 characters:      {(text_df['characters'] < 1000).sum()}")

print("\nText length statistics:")
display(text_df["characters"].describe())

=== TEXT INTEGRITY ===
Empty documents:        0
<100 characters:        0
<500 characters:        4
<1,000 characters:      33

Text length statistics:


count       300.000000
mean      23457.540000
std       43892.558448
min         434.000000
25%        2993.500000
50%       11834.000000
75%       32370.500000
max      611489.000000
Name: characters, dtype: float64

In [10]:
short_docs = (
    text_df
    .sort_values("characters")
    .head(20)
)

display(short_docs)

,document_id,title,characters,words,parts
283,5739890,მნიშვნელოვანი ქონებით უზრუნველყოფილი ფიზიკური ...,434,45,5
278,5762493,„ტურისტული მომსახურების დღგ-ით დაბეგვრის თაობა...,478,52,5
142,6231728,ბუღალტრისა და აუდიტორის პროფესიული დღის განსაზ...,483,51,6
282,5749638,„ყულევის თავისუფალი ინდუსტრიული ზონის შექმნის ...,497,52,5
100,6562716,„საქართველოს საჯარო სექტორის ბუღალტრული აღრიცხ...,523,54,5
202,5958945,“ელექტრონული კომერციის შესახებ” საქართველოს კა...,562,63,5
11,6933818,„სახელმწიფო სამეცნიერო გრანტების გაცემის შესახ...,597,62,5
38,6810516,საინვესტიციო ოქროს სტატუსის დადგენის დამატებით...,610,70,5
26,6865118,„სს „საქართველოს განვითარების ფონდის“ წესდების...,635,68,5
34,6827122,საქართველოს ეროვნული ბანკის ზედამხედველობას და...,686,76,5


# Integrity

In [11]:
print("=== PART INTEGRITY ===")

zero_parts = [
    d for d in docs
    if not d.get("parts")
]

zero_part_count = len(zero_parts)

empty_parts = []

for d in docs:
    for part in d.get("parts", []):
        if not (part.get("text") or "").strip():
            empty_parts.append({
                "document_id": d["document_id"],
                "title": d["title"],
                "part_id": part.get("id"),
                "type": part.get("type"),
                "structure": part.get("structure"),
            })

print(f"Documents with zero parts: {zero_part_count}")
print(f"Empty parts:               {len(empty_parts)}")

if empty_parts:
    display(pd.DataFrame(empty_parts).head(50))

=== PART INTEGRITY ===
Documents with zero parts: 0
Empty parts:               689


,document_id,title,part_id,type,structure
0,7010313,საქართველოს ეროვნული ბანკის 2026-2027 წლების ა...,DOCUMENT:1;PREAMBLE:1;_Content,PREAMBLE,"[DOCUMENT:1, PREAMBLE:1]"
1,7010249,სახელმწიფო აუდიტის სამსახურის 2026-2027 წლების...,DOCUMENT:1;PREAMBLE:1;_Content,PREAMBLE,"[DOCUMENT:1, PREAMBLE:1]"
2,7002597,ადმინისტრაციული სახდელისგან გათავისუფლების შეს...,DOCUMENT:1;PREAMBLE:1;_Content,PREAMBLE,"[DOCUMENT:1, PREAMBLE:1]"
3,6987795,სახელმწიფო შესყიდვის დარეზერვების წესისა და პი...,DOCUMENT:1;PREAMBLE:1;_Content,PREAMBLE,"[DOCUMENT:1, PREAMBLE:1]"
4,6987795,სახელმწიფო შესყიდვის დარეზერვების წესისა და პი...,DOCUMENT:1;ENCLOSURE:1;PREAMBLE:1;_Content,ENCLOSURE,"[DOCUMENT:1, ENCLOSURE:1, PREAMBLE:1]"
5,6987795,სახელმწიფო შესყიდვის დარეზერვების წესისა და პი...,DOCUMENT:1;ENCLOSURE:1;FOOTER:1;_Content,ENCLOSURE,"[DOCUMENT:1, ENCLOSURE:1, FOOTER:1]"
6,6981470,ვირტუალური აქტივის მომსახურების პროვაიდერის მი...,DOCUMENT:1;ENCLOSURE:1;HEADER:1;_Content,ENCLOSURE,"[DOCUMENT:1, ENCLOSURE:1, HEADER:1]"
7,6981470,ვირტუალური აქტივის მომსახურების პროვაიდერის მი...,DOCUMENT:1;ENCLOSURE:1;PREAMBLE:1;_Content,ENCLOSURE,"[DOCUMENT:1, ENCLOSURE:1, PREAMBLE:1]"
8,6981470,ვირტუალური აქტივის მომსახურების პროვაიდერის მი...,DOCUMENT:1;ENCLOSURE:1;FOOTER:1;_Content,ENCLOSURE,"[DOCUMENT:1, ENCLOSURE:1, FOOTER:1]"
9,6969117,ფასიანი ქაღალდების რეგისტრატორის და საბროკერო ...,DOCUMENT:1;ENCLOSURE:1;PREAMBLE:1;_Content,ENCLOSURE,"[DOCUMENT:1, ENCLOSURE:1, PREAMBLE:1]"


# Structure

In [12]:
part_types = Counter()

for d in docs:
    for part in d.get("parts", []):
        part_types[part.get("type")] += 1

print("=== PART TYPES ===")

part_type_df = (
    pd.DataFrame(
        part_types.items(),
        columns=["type", "count"]
    )
    .sort_values("count", ascending=False)
)

display(part_type_df)

=== PART TYPES ===


,type,count
5,ENCLOSURE,1460
4,ARTICLE,845
0,HEADER,300
3,FOOTER,300
1,PREAMBLE,300
6,CHAPTER,174
2,POINT,24


In [13]:
enclosure_parts = []

for d in docs:
    for part in d.get("parts", []):
        if part.get("enclosure") is not None:
            enclosure_parts.append({
                "document_id": d["document_id"],
                "title": d["title"],
                "part_id": part.get("id"),
                "type": part.get("type"),
                "enclosure": part.get("enclosure"),
                "structure": " > ".join(part.get("structure", [])),
                "characters": len(part.get("text") or ""),
            })

print(f"Parts belonging to enclosures: {len(enclosure_parts)}")

if enclosure_parts:
    enclosure_df = pd.DataFrame(enclosure_parts)
    display(enclosure_df.head(30))

Parts belonging to enclosures: 1460


,document_id,title,part_id,type,enclosure,structure,characters
0,6987795,სახელმწიფო შესყიდვის დარეზერვების წესისა და პი...,DOCUMENT:1;ENCLOSURE:1;HEADER:1;_Content,ENCLOSURE,1,DOCUMENT:1 > ENCLOSURE:1 > HEADER:1,50
1,6987795,სახელმწიფო შესყიდვის დარეზერვების წესისა და პი...,DOCUMENT:1;ENCLOSURE:1;PREAMBLE:1;_Content,ENCLOSURE,1,DOCUMENT:1 > ENCLOSURE:1 > PREAMBLE:1,0
2,6987795,სახელმწიფო შესყიდვის დარეზერვების წესისა და პი...,DOCUMENT:1;ENCLOSURE:1;ARTICLE:1;_Content,ENCLOSURE,1,DOCUMENT:1 > ENCLOSURE:1 > ARTICLE:1,8079
3,6987795,სახელმწიფო შესყიდვის დარეზერვების წესისა და პი...,DOCUMENT:1;ENCLOSURE:1;FOOTER:1;_Content,ENCLOSURE,1,DOCUMENT:1 > ENCLOSURE:1 > FOOTER:1,0
4,6981470,ვირტუალური აქტივის მომსახურების პროვაიდერის მი...,DOCUMENT:1;ENCLOSURE:1;HEADER:1;_Content,ENCLOSURE,1,DOCUMENT:1 > ENCLOSURE:1 > HEADER:1,0
5,6981470,ვირტუალური აქტივის მომსახურების პროვაიდერის მი...,DOCUMENT:1;ENCLOSURE:1;PREAMBLE:1;_Content,ENCLOSURE,1,DOCUMENT:1 > ENCLOSURE:1 > PREAMBLE:1,0
6,6981470,ვირტუალური აქტივის მომსახურების პროვაიდერის მი...,DOCUMENT:1;ENCLOSURE:1;ARTICLE:1;_Content,ENCLOSURE,1,DOCUMENT:1 > ENCLOSURE:1 > ARTICLE:1,8611
7,6981470,ვირტუალური აქტივის მომსახურების პროვაიდერის მი...,DOCUMENT:1;ENCLOSURE:1;FOOTER:1;_Content,ENCLOSURE,1,DOCUMENT:1 > ENCLOSURE:1 > FOOTER:1,0
8,6969117,ფასიანი ქაღალდების რეგისტრატორის და საბროკერო ...,DOCUMENT:1;ENCLOSURE:1;HEADER:1;_Content,ENCLOSURE,1,DOCUMENT:1 > ENCLOSURE:1 > HEADER:1,278
9,6969117,ფასიანი ქაღალდების რეგისტრატორის და საბროკერო ...,DOCUMENT:1;ENCLOSURE:1;PREAMBLE:1;_Content,ENCLOSURE,1,DOCUMENT:1 > ENCLOSURE:1 > PREAMBLE:1,0


# HTML Leakage

In [14]:
html_patterns = [
    r"<[a-zA-Z][^>]*>",
    r"</[a-zA-Z]+>",
    r"&nbsp;",
    r"&amp;",
]

html_leaks = []

for d in docs:
    text = d.get("text") or ""
    
    matches = []
    
    for pattern in html_patterns:
        matches.extend(re.findall(pattern, text))
    
    if matches:
        html_leaks.append({
            "document_id": d["document_id"],
            "title": d["title"],
            "matches": matches[:10],
        })

print(f"Documents with possible HTML leakage: {len(html_leaks)}")

if html_leaks:
    display(pd.DataFrame(html_leaks).head(30))

Documents with possible HTML leakage: 0


# Georgian chars

In [15]:
def georgian_char_count(text):
    return len(re.findall(r"[ა-ჰ]", text))

def latin_char_count(text):
    return len(re.findall(r"[A-Za-z]", text))

def cyrillic_char_count(text):
    return len(re.findall(r"[А-Яа-я]", text))


script_rows = []

for d in docs:
    text = d.get("text") or ""
    
    script_rows.append({
        "document_id": d["document_id"],
        "title": d["title"],
        "characters": len(text),
        "georgian": georgian_char_count(text),
        "latin": latin_char_count(text),
        "cyrillic": cyrillic_char_count(text),
    })

script_df = pd.DataFrame(script_rows)

script_df["georgian_ratio"] = (
    script_df["georgian"] /
    script_df["characters"].replace(0, 1)
)

print("=== SCRIPT SANITY ===")

print(
    f"Documents with zero Georgian characters: "
    f"{(script_df['georgian'] == 0).sum()}"
)

print(
    f"Documents with <1% Georgian characters: "
    f"{(script_df['georgian_ratio'] < 0.01).sum()}"
)

display(
    script_df
    .sort_values("georgian_ratio")
    .head(20)
)

=== SCRIPT SANITY ===
Documents with zero Georgian characters: 0
Documents with <1% Georgian characters: 0


,document_id,title,characters,georgian,latin,cyrillic,georgian_ratio
207,5936431,მსოფლიო ბანკთაშორისი საფინანსო ტელეკომუნიკაციე...,12363,7335,2873,0,0.593303
129,6320610,საჯარო სამართლის იურიდიული პირის − საქართველოს...,69068,47705,11050,0,0.690696
80,6703385,აჭარის ავტონომიური რესპუბლიკის კანონი აჭარის ა...,129739,91892,38,0,0.708284
159,6156028,საზოგადოებრივი ჯანმრთელობის დელეგირებული უფლებ...,8197,5864,232,2,0.715384
79,6710623,აფხაზეთის ავტონომიური რესპუბლიკის 2026 წლის რე...,86196,64194,49,0,0.744745
1,7010249,სახელმწიფო აუდიტის სამსახურის 2026-2027 წლების...,894,682,27,0,0.762864
83,6701007,საქართველოს 2026 წლის სახელმწიფო ბიუჯეტის შესახებ,611489,467965,1872,2,0.765288
148,6206301,საქართველოს ეროვნული ბანკის 2024-2025 წლების ა...,1004,772,27,0,0.768924
91,6629760,დეპოზიტორისთვის ასანაზღაურებელი დაზღვეული დეპო...,1029,794,14,0,0.771623
62,6737639,საჯარო სამართლის იურიდიული პირის - შემოსავლები...,2164,1678,0,0,0.775416


# Document types

In [16]:
print("=== DOCUMENT TYPES ===")

type_df = (
    pd.DataFrame(
        Counter(d.get("document_type") for d in docs).items(),
        columns=["document_type", "count"]
    )
    .sort_values("count", ascending=False)
)

display(type_df)

=== DOCUMENT TYPES ===


,document_type,count
2,საქართველოს მთავრობის დადგენილება,104
3,საქართველოს ეროვნული ბანკის პრეზიდენტის ბრძანება,90
6,საქართველოს მინისტრის ბრძანება,33
7,"ბუღალტრული აღრიცხვის, ანგარიშგებისა და აუდიტის...",20
1,საქართველოს კანონი,10
0,საქართველოს პარლამენტის დადგენილება,8
15,საქართველოს ეროვნული ბანკის საბჭოს დადგენილება,7
4,სახელმწიფო აუდიტის სამსახურის გენერალური აუდიტ...,5
8,სახელმწიფო შესყიდვების სააგენტოს თავმჯდომარის ...,4
13,სსიპ დეპოზიტების დაზღვევის სააგენტოს უფროსის ბ...,4


In [17]:
print("=== ISSUING AUTHORITIES ===")

authority_df = (
    pd.DataFrame(
        Counter(d.get("issuing_authority") for d in docs).items(),
        columns=["issuing_authority", "count"]
    )
    .sort_values("count", ascending=False)
)

display(authority_df.head(30))

=== ISSUING AUTHORITIES ===


,issuing_authority,count
1,საქართველოს მთავრობა,104
2,საქართველოს ეროვნული ბანკის პრეზიდენტი,90
5,საქართველოს ფინანსთა მინისტრი,29
6,"ბუღალტრული აღრიცხვის, ანგარიშგებისა და აუდიტის...",20
0,საქართველოს პარლამენტი,18
16,საქართველოს ეროვნული ბანკის საბჭო,7
3,სახელმწიფო აუდიტის სამსახურის გენერალური აუდიტორი,5
14,სსიპ დეპოზიტების დაზღვევის სააგენტო,4
7,სახელმწიფო შესყიდვების სააგენტოს თავმჯდომარე,4
18,აჭარის ავტონომიური რესპუბლიკის ფინანსთა და ეკო...,3


In [18]:
from datetime import datetime

bad_dates = []
parsed_dates = []

for d in docs:
    date_string = d.get("signing_date")
    
    try:
        parsed = datetime.strptime(date_string, "%d/%m/%Y")
        parsed_dates.append(parsed)
    except (TypeError, ValueError):
        bad_dates.append({
            "document_id": d["document_id"],
            "signing_date": date_string,
            "title": d["title"],
        })

print("=== DATE SANITY ===")
print(f"Bad/unparseable dates: {len(bad_dates)}")

if bad_dates:
    display(pd.DataFrame(bad_dates))

if parsed_dates:
    print(f"Earliest signing date: {min(parsed_dates).date()}")
    print(f"Latest signing date:   {max(parsed_dates).date()}")

=== DATE SANITY ===
Bad/unparseable dates: 0
Earliest signing date: 2022-12-23
Latest signing date:   2026-09-29


# Random Check

In [19]:
import random

random.seed(42)

sample = random.sample(docs, min(10, len(docs)))

for i, d in enumerate(sample, start=1):
    print("=" * 100)
    print(f"SAMPLE {i}")
    print(f"ID:       {d['document_id']}")
    print(f"Title:    {d['title']}")
    print(f"Type:     {d['document_type']}")
    print(f"Authority:{d['issuing_authority']}")
    print(f"Date:     {d['signing_date']}")
    print(f"Parts:    {d['part_count']}")
    print(f"Chars:    {len(d['text'])}")
    print()
    print(d["text"][:1500])
    print()

SAMPLE 1
ID:       6733900
Title:    შპს საქართველოს ფოსტის მიერ საქონლისა და მომსახურების შესყიდვების სპეციალური წესის დამტკიცების შესახებ
Type:     საქართველოს მთავრობის დადგენილება
Authority:საქართველოს მთავრობა
Date:     29/12/2025
Parts:    10
Chars:    32014

შპს საქართველოს ფოსტის მიერ საქონლისა და მომსახურების შესყიდვების სპეციალური წესის დამტკიცების შესახებ

„სახელმწიფო შესყიდვების შესახებ“ საქართველოს კანონის მე-3 მუხლის პირველი პუნქტის „ა.თ“ ქვეპუნქტის შესაბამისად, დამტკიცდეს თანდართული „შპს საქართველოს ფოსტის მიერ საქონლისა და მომსახურების შესყიდვების სპეციალური წესი“.

ეს დადგენილება მოქმედია 2026 წლის 31 დეკემბრის ჩათვლით.

დადგენილება ამოქმედდეს 2026 წლის 1 იანვრიდან.

პრემიერ-მინისტრი ირაკლი კობახიძე

მუხლი 1. ზოგადი დებულებები 1. „შპს საქართველოს ფოსტის მიერ საქონლისა და მომსახურების შესყიდვების სპეციალური წესი“ (შემდგომში – წესი) შემუშავებულია „სახელმწიფო შესყიდვების შესახებ“ საქართველოს კანონის მე-3 მუხლის პირველი პუნქტის „ა.თ“ ქვეპუნქტის შესაბამისად და ადგენს ამ წეს

In [20]:
enclosure_doc_ids = sorted({
    row["document_id"]
    for row in enclosure_parts
})

print(f"Documents with enclosures: {len(enclosure_doc_ids)}")

for document_id in enclosure_doc_ids[:5]:
    d = next(x for x in docs if x["document_id"] == document_id)
    
    print("=" * 100)
    print(d["document_id"])
    print(d["title"])
    
    for part in d["parts"]:
        print(
            f"\nTYPE={part['type']} | "
            f"ENCLOSURE={part['enclosure']} | "
            f"STRUCTURE={' > '.join(part['structure'])}"
        )
        print(part["text"][:500])

Documents with enclosures: 195
5662765
საქართველოს გარემოს დაცვისა და სოფლის მეურნეობის სამინისტროს სისტემის უფლებამოსილი ორგანოს მიერ ცხოველის ან/და მცენარის ახალი ჯიშის განსხვავებულობაზე, ერთგვაროვნებასა და სტაბილურობაზე გამოცდის ჩატარების საფასურის ოდენობის დამტკიცების შესახებ

TYPE=HEADER | ENCLOSURE=None | STRUCTURE=DOCUMENT:1 > HEADER:1
საქართველოს გარემოს დაცვისა და სოფლის მეურნეობის სამინისტროს სისტემის უფლებამოსილი ორგანოს მიერ ცხოველის ან/და მცენარის ახალი ჯიშის განსხვავებულობაზე, ერთგვაროვნებასა და სტაბილურობაზე გამოცდის ჩატარების საფასურის ოდენობის დამტკიცების შესახებ

TYPE=PREAMBLE | ENCLOSURE=None | STRUCTURE=DOCUMENT:1 > PREAMBLE:1


TYPE=ARTICLE | ENCLOSURE=None | STRUCTURE=DOCUMENT:1 > ARTICLE:1
„ცხოველთა და მცენარეთა ახალი ჯიშების შესახებ“ საქართველოს კანონის 47-ე მუხლის მე-6 პუნქტის „ბ“ ქვეპუნქტის შესაბამისად, დამტკიცდეს თანდართული „საქართველოს გარემოს დაცვისა და სოფლის მეურნეობის სამინისტროს სისტემის უფლებამოსილი ორგანოს მიერ ცხოველის ან/და მცენარის ახალი ჯიშის განს

# Inspect Structure

In [44]:
from collections import Counter

structure_patterns = Counter()

for d in docs:
    pattern = tuple(
        part["type"]
        for part in d.get("parts", [])
        if part.get("text")  # only non-empty parts
    )
    structure_patterns[pattern] += 1

for pattern, count in structure_patterns.most_common():
    print(f"{count:3} docs | " + " → ".join(pattern))


    unique_types = sorted({
    part["type"]
    for d in docs
    for part in d.get("parts", [])
    if part.get("text")
})



 27 docs | HEADER → PREAMBLE → ARTICLE → ARTICLE → FOOTER
 19 docs | HEADER → ARTICLE → ARTICLE → FOOTER → ENCLOSURE → ENCLOSURE
 19 docs | HEADER → PREAMBLE → ARTICLE → ARTICLE → ARTICLE → FOOTER → ENCLOSURE
 16 docs | HEADER → POINT → FOOTER
 16 docs | HEADER → ARTICLE → ARTICLE → ARTICLE → FOOTER → ENCLOSURE → ENCLOSURE
 16 docs | HEADER → PREAMBLE → ARTICLE → ARTICLE → FOOTER → ENCLOSURE
 16 docs | HEADER → PREAMBLE → ARTICLE → ARTICLE → ARTICLE → FOOTER
 14 docs | HEADER → ARTICLE → ARTICLE → FOOTER → ENCLOSURE
 13 docs | HEADER → ARTICLE → ARTICLE → FOOTER
 13 docs | HEADER → ARTICLE → ARTICLE → ARTICLE → FOOTER → ENCLOSURE
 10 docs | HEADER → PREAMBLE → ARTICLE → ARTICLE → FOOTER → ENCLOSURE → ENCLOSURE
 10 docs | HEADER → PREAMBLE → ARTICLE → ARTICLE → ARTICLE → FOOTER → ENCLOSURE → ENCLOSURE
  7 docs | HEADER → PREAMBLE → ARTICLE → ARTICLE → ARTICLE → ARTICLE → FOOTER
  6 docs | HEADER → PREAMBLE → ARTICLE → ARTICLE → ARTICLE → ARTICLE → FOOTER → ENCLOSURE
  5 docs | HEADER → 

In [43]:
print(unique_types)

['ARTICLE', 'CHAPTER', 'ENCLOSURE', 'FOOTER', 'HEADER', 'POINT', 'PREAMBLE']


In [40]:
for target_type in ["FOOTER", "POINT", "PREAMBLE"]:
    print(f"\n{'=' * 80}")
    print(f"{target_type}")
    print(f"{'=' * 80}")

    count = 0

    for d in docs:
        for part in d.get("parts", []):
            if part.get("type") == target_type and part.get("text"):
                print(f"\nDocument ID: {d['document_id']}")
                print(f"Title: {d['title']}")
                print(f"Structure: {' → '.join(part['structure'])}")
                print(f"Text: {part['text']}")

                count += 1

                if count == 3:
                    break

        if count == 3:
            break


FOOTER

Document ID: 7010313
Title: საქართველოს ეროვნული ბანკის 2026-2027 წლების ანგარიშგების გარე აუდიტის განსახორციელებლად დამოუკიდებელი აუდიტორული ფირმის (გარე აუდიტორის) არჩევის შესახებ
Structure: DOCUMENT:1 → FOOTER:1
Text: საქართველოს პარლამენტის თავმჯდომარე შალვა პაპუაშვილი თბილისი, 29 სექტემბერი 2026 წ. N1872-VIმს-XIმპ

Document ID: 7010249
Title: სახელმწიფო აუდიტის სამსახურის 2026-2027 წლების ფინანსური აუდიტის განსახორციელებლად აუდიტორული ფირმის არჩევის შესახებ
Structure: DOCUMENT:1 → FOOTER:1
Text: საქართველოს პარლამენტის თავმჯდომარე შალვა პაპუაშვილი თბილისი, 29 სექტემბერი 2026 წ. N1871-VIმს-XIმპ

Document ID: 7002597
Title: ადმინისტრაციული სახდელისგან გათავისუფლების შესახებ
Structure: DOCUMENT:1 → FOOTER:1
Text: საქართველოს პრეზიდენტი მიხეილ ყაველაშვილი თბილისი, 16 სექტემბერი 2026 წ. N1861/5-VIმს-XIმპ

POINT

Document ID: 7010313
Title: საქართველოს ეროვნული ბანკის 2026-2027 წლების ანგარიშგების გარე აუდიტის განსახორციელებლად დამოუკიდებელი აუდიტორული ფირმის (გარე აუდიტორის) ა

In [39]:
import re
from collections import defaultdict

unique_types = sorted({
    part["type"]
    for d in docs
    for part in d.get("parts", [])
    if part.get("text")
})

# Georgian structural terms to search for
georgian_terms = {
    "მუხლი": "ARTICLE",
    "პუნქტი": "POINT",
    "თავი": "CHAPTER",
    "დანართი": "ENCLOSURE",
}

type_georgian = {}

for part_type in unique_types:
    matches = []

    for d in docs:
        for part in d.get("parts", []):
            if part.get("type") != part_type or not part.get("text"):
                continue

            text = part["text"]

            for georgian_name, mapped_type in georgian_terms.items():
                if mapped_type == part_type and re.search(
                    rf"(?<!\S){re.escape(georgian_name)}(?!\S)", text
                ):
                    matches.append(georgian_name)

    if matches:
        type_georgian[part_type] = max(set(matches), key=matches.count)

print(type_georgian)

{'ARTICLE': 'მუხლი', 'CHAPTER': 'თავი', 'ENCLOSURE': 'დანართი', 'POINT': 'პუნქტი'}


In [23]:
structure_counts = Counter()

for d in docs:
    for part in d.get("parts", []):
        enclosure = part.get("enclosure")

        if enclosure is None:
            location = "MAIN"
        else:
            location = f"ANNEX_{enclosure}"

        structure_counts[(location, part["type"])] += 1

for (location, part_type), count in structure_counts.most_common():
    print(f"{location:12} | {part_type:15} | {count:4}")

ANNEX_1      | ENCLOSURE       | 1378
MAIN         | ARTICLE         |  845
MAIN         | HEADER          |  300
MAIN         | PREAMBLE        |  300
MAIN         | FOOTER          |  300
MAIN         | CHAPTER         |  174
ANNEX_2      | ENCLOSURE       |   66
MAIN         | POINT           |   24
ANNEX_3      | ENCLOSURE       |    4
ANNEX_4      | ENCLOSURE       |    4
ANNEX_5      | ENCLOSURE       |    4
ANNEX_6      | ENCLOSURE       |    4


direct parent-child

In [45]:
from collections import Counter

relationships = Counter()

for d in docs:
    for part in d.get("parts", []):
        structure = part.get("structure", [])

        for parent, child in zip(structure, structure[1:]):
            parent_type = parent.split(":", 1)[0]
            child_type = child.split(":", 1)[0]

            relationships[(parent_type, child_type)] += 1

for (parent, child), count in relationships.most_common():
    print(f"{count:4} | {parent} → {child}")

1460 | DOCUMENT → ENCLOSURE
 845 | DOCUMENT → ARTICLE
 642 | ENCLOSURE → ARTICLE
 313 | CHAPTER → ARTICLE
 300 | DOCUMENT → HEADER
 300 | DOCUMENT → PREAMBLE
 300 | DOCUMENT → FOOTER
 213 | ENCLOSURE → HEADER
 213 | ENCLOSURE → PREAMBLE
 213 | ENCLOSURE → FOOTER
 174 | DOCUMENT → CHAPTER
 139 | ENCLOSURE → CHAPTER
  75 | ARTICLE → POINT
  40 | ENCLOSURE → POINT
  24 | DOCUMENT → POINT


In [24]:
def print_structure(doc):
    print("=" * 100)
    print(doc["document_id"])
    print(doc["title"])
    print()

    for part in doc["parts"]:
        structure = " → ".join(part["structure"])
        chars = len(part.get("text", ""))

        print(
            f"{part['type']:12} | "
            f"annex={part['enclosure']!s:4} | "
            f"{chars:7,} chars | "
            f"{structure}"
        )

In [25]:
print_structure(docs[0])

7010313
საქართველოს ეროვნული ბანკის 2026-2027 წლების ანგარიშგების გარე აუდიტის განსახორციელებლად დამოუკიდებელი აუდიტორული ფირმის (გარე აუდიტორის) არჩევის შესახებ

HEADER       | annex=None |     153 chars | DOCUMENT:1 → HEADER:1
PREAMBLE     | annex=None |       0 chars | DOCUMENT:1 → PREAMBLE:1
POINT        | annex=None |     758 chars | DOCUMENT:1 → POINT:1
FOOTER       | annex=None |      99 chars | DOCUMENT:1 → FOOTER:1


In [26]:
complexity = []

for d in docs:
    parts = d.get("parts", [])

    complexity.append({
        "document_id": d["document_id"],
        "title": d["title"],
        "parts": len(parts),
        "nonempty_parts": sum(bool(p.get("text")) for p in parts),
        "annexes": len({
            p["enclosure"]
            for p in parts
            if p.get("enclosure") is not None
        }),
        "characters": len(d.get("text", "")),
    })

complexity_df = pd.DataFrame(complexity)

display(
    complexity_df
    .sort_values(["annexes", "parts", "characters"], ascending=False)
    .head(20)
)

,document_id,title,parts,nonempty_parts,annexes,characters
9,6952872,სახელმწიფო აუდიტის სამსახურის სამსახურებრივ-სა...,34,18,6,10783
247,5834965,მიკრობანკების კაპიტალის ადეკვატურობის მოთხოვნე...,74,68,2,101718
228,5840866,მიკრობანკების კიბერუსაფრთხოების მართვის ჩარჩოს...,29,24,2,32510
207,5936431,მსოფლიო ბანკთაშორისი საფინანსო ტელეკომუნიკაციე...,21,15,2,12363
210,5920196,აჭარის ავტონომიური რესპუბლიკის რესპუბლიკური ბი...,16,11,2,12817
110,6471596,"ენერგოაუდიტორების, ენერგოეფექტურობის სერტიფიკა...",15,8,2,54030
179,6103380,კომერციული ბანკის ლიკვიდაციისა და გადახდისუუნა...,15,9,2,53735
178,6103416,მიკროსაფინანსო ორგანიზაციის ლიკვიდაციის წესის ...,15,9,2,42063
222,5874397,აფხაზეთის ავტონომიური რესპუბლიკის ერთიანი სააღ...,14,8,2,123244
133,6279133,საქართველოს ეროვნული ბანკის მიერ ბოლო ინსტანცი...,14,10,2,17308


check docs that have a lot of annexes

In [27]:
max_annex_doc = max(
    docs,
    key=lambda d: len({
        p["enclosure"]
        for p in d.get("parts", [])
        if p.get("enclosure") is not None
    })
)

print_structure(max_annex_doc)

6952872
სახელმწიფო აუდიტის სამსახურის სამსახურებრივ-საშტატო ცეცხლსასროლი იარაღის შენახვის, გაცემისა და აღრიცხვის წესის განსაზღვრის შესახებ

HEADER       | annex=None |     130 chars | DOCUMENT:1 → HEADER:1
PREAMBLE     | annex=None |     461 chars | DOCUMENT:1 → PREAMBLE:1
ARTICLE      | annex=None |     132 chars | DOCUMENT:1 → ARTICLE:1
ARTICLE      | annex=None |      43 chars | DOCUMENT:1 → ARTICLE:2
FOOTER       | annex=None |      68 chars | DOCUMENT:1 → FOOTER:1
ENCLOSURE    | annex=1    |       0 chars | DOCUMENT:1 → ENCLOSURE:1 → HEADER:1
ENCLOSURE    | annex=1    |       0 chars | DOCUMENT:1 → ENCLOSURE:1 → PREAMBLE:1
ENCLOSURE    | annex=1    |   1,725 chars | DOCUMENT:1 → ENCLOSURE:1 → ARTICLE:1
ENCLOSURE    | annex=1    |     444 chars | DOCUMENT:1 → ENCLOSURE:1 → ARTICLE:2
ENCLOSURE    | annex=1    |   1,527 chars | DOCUMENT:1 → ENCLOSURE:1 → ARTICLE:3
ENCLOSURE    | annex=1    |   1,807 chars | DOCUMENT:1 → ENCLOSURE:1 → ARTICLE:4
ENCLOSURE    | annex=1    |   1,818 char

In [28]:
print(max_annex_doc["document_id"])
print(max_annex_doc["title"])

6952872
სახელმწიფო აუდიტის სამსახურის სამსახურებრივ-საშტატო ცეცხლსასროლი იარაღის შენახვის, გაცემისა და აღრიცხვის წესის განსაზღვრის შესახებ


# Chunking logic

* Preserve the original Matsne document hierarchy and structure.
* Treat **Article** as the primary chunking unit whenever Articles exist.
* Keep all **Points** inside their parent Article; do not split Points into separate chunks.
* If a **Chapter contains Articles**, create one chunk per Article and keep the Chapter as hierarchical context.
* If a **Chapter does not contain Articles** but contains substantive content, use the Chapter as the chunk.
* Treat **Header** as contextual information and attach it to the relevant substantive chunks within the same structural branch.
* Treat **Preamble** as contextual information and attach it to the relevant Chapter/Article chunks within the same structural branch.

  * `DOCUMENT → PREAMBLE` applies to the relevant main-document chunks.
  * `ENCLOSURE → PREAMBLE` applies to the chunks within that enclosure.
* Treat **Footer** as metadata/provenance rather than retrieval content.
* Treat **Enclosure** as a structural container, not as a chunk by itself. Its Articles/Chapters are chunked using the same rules.
* Preserve the full hierarchy in each chunk's metadata, e.g. document → enclosure → chapter → article.
* Keep the **original text unchanged**; contextual information can be added to a separate `retrieval_text` used for embedding/retrieval.
* If a resulting legal unit is exceptionally large, only then apply a secondary size-based split while preserving its legal hierarchy and context.

# Extracting links to generate evaluation set

In [54]:
# ============================================================
# EXTRACT DOCUMENT NAMES + LINKS
# ============================================================

import csv
from urllib.parse import urljoin

# ============================================================
# CONFIGURATION
# ============================================================

BASE_URL = "https://matsne.gov.ge"
SEARCH_URL = f"{BASE_URL}/ka/document/search"

MAX_PAGES = 5
RESULTS_PER_PAGE = 100
REQUEST_DELAY = 1.0

OUTPUT_DIR = Path("../matsne_data")
RAW_SEARCH_DIR = OUTPUT_DIR / "search_pages"

RAW_SEARCH_DIR.mkdir(parents=True, exist_ok=True)

DOCUMENTS_FILE = OUTPUT_DIR / "documents.csv"

MAX_PAGES = 3

def extract_documents_from_page(html):
    soup = BeautifulSoup(html, "html.parser")

    documents = []

    # Find links that point to Matsne document pages
    for link in soup.find_all("a", href=True):

        href = link["href"]

        # Keep only actual document links
        if "/ka/document/view/" not in href:
            continue

        name = link.get_text(" ", strip=True)

        if not name:
            continue

        url = urljoin(BASE_URL, href)

        documents.append({
            "name": name,
            "url": url
        })

    return documents


# ============================================================
# EXTRACT FROM ALL SAVED SEARCH PAGES
# ============================================================

all_documents = []

for page in range(1, MAX_PAGES + 1):

    html_file = RAW_SEARCH_DIR / f"page_{page}.html"

    if not html_file.exists():
        print(f"Missing: {html_file}")
        continue

    html = html_file.read_text(
        encoding="utf-8"
    )

    page_documents = extract_documents_from_page(html)

    print(
        f"Page {page}: "
        f"{len(page_documents)} document links found"
    )

    all_documents.extend(page_documents)


# ============================================================
# REMOVE DUPLICATES
# ============================================================

unique_documents = {}

for document in all_documents:

    unique_documents[document["url"]] = document


all_documents = list(unique_documents.values())


# ============================================================
# SAVE CSV
# ============================================================

with open(
    DOCUMENTS_FILE,
    "w",
    encoding="utf-8-sig",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=["name", "url"]
    )

    writer.writeheader()

    writer.writerows(all_documents)


# ============================================================
# SUMMARY
# ============================================================

print("\nExtraction complete.")
print(
    f"Unique documents found: "
    f"{len(all_documents)}"
)

print(
    f"Saved to: {DOCUMENTS_FILE}"
)

Page 1: 106 document links found
Page 2: 118 document links found
Page 3: 118 document links found

Extraction complete.
Unique documents found: 342
Saved to: ../matsne_data/documents.csv


In [55]:
eval_df = pd.read_csv("../evaluation/eval_questions.csv")

In [58]:
eval_df

,კითხვა,ოქროს_დოკუმენტი,დოკუმენტის_ID,წყარო,პასუხი
0,თუ საქართველოს ეროვნულ ბანკს 2026–2027 წლების ...,საქართველოს ეროვნული ბანკის 2026-2027 წლების ა...,7010313,"მუხლი 1, პუნქტები 1–5",გარე აუდიტორად არჩეულია შპს „დელოიტი და ტუში“....
1,რა განსხვავებაა სახელმწიფო აუდიტის სამსახურის ...,სახელმწიფო აუდიტის სამსახურის 2026-2027 წლების...,7010249,"მუხლი 1, პუნქტები 1–5",აუდიტორულ ფირმად არჩეულია შპს „KPMG Georgia“. ...
2,სახელმწიფო შესყიდვის დარეზერვებისას როგორ განი...,სახელმწიფო შესყიდვის დარეზერვების წესისა და პი...,6987795,"მუხლი 1, მუხლი 2",ეკონომიკურ ოპერატორთა წრე განისაზღვრება თავად ...
3,რა ინფორმაციის შეფასებას ემსახურება სტაბილური ...,ვირტუალური აქტივის მომსახურების პროვაიდერის მი...,6981470,"მუხლი 1, პუნქტები 2–3; მუხლი 2",ანგარიშგების მიზანია შეფასდეს ემიტენტის მიერ გ...
4,ფასიანი ქაღალდების რეგისტრატორის ან საბროკერო ...,ფასიანი ქაღალდების რეგისტრატორის და საბროკერო ...,6969117,"მუხლი 1, პუნქტები 8–9; მუხლი 2",გადახდილი ფულადი ჯარიმა მიემართება საქართველოს...
5,თუ პირს ფასიანი ქაღალდების ან ბუღალტრული აღრიც...,პირების მიმართ ფასიანი ქაღალდების შესახებ კანო...,6969072,"მუხლი 1, მუხლი 2","არა. დოკუმენტი პირდაპირ ადგენს, რომ ლიცენზიის,..."
6,რა არის საინვესტიციო ფონდების მხარდაჭერის სახე...,„საინვესტიციო ფონდების მხარდაჭერის სახელმწიფო ...,6961997,მუხლი 2; მუხლი 3,პროგრამის მიზანია საინვესტიციო ფონდების დაფუძნ...
7,აფხაზეთის ავტონომიური რესპუბლიკის რესპუბლიკური...,აფხაზეთის ავტონომიური რესპუბლიკის რესპუბლიკური...,6951831,მუხლი 1,ბრძანებით მტკიცდება აფხაზეთის ავტონომიური რესპ...
8,აგროთანადაფინანსების სახელმწიფო პროგრამის შემთ...,აგროთანადაფინანსების სახელმწიფო პროგრამის დამტ...,6906141,მუხლი 1; მუხლი 2; დანართი,დადგენილებით მტკიცდება თანდართული „აგროთანადაფ...
9,რა ცვლილებას ახდენს ეროვნული ბანკის 2026 წლის ...,სესხის გამცემი ორგანიზაციების მიერ 1 000 000 ლ...,6892788,მუხლი 1; მუხლი 2; მუხლი 3,ახალი ბრძანება ეხება სესხის გამცემი ორგანიზაცი...
